#large language model data analysis

We are gonna look at things like cross model heterogeneity, differential treatment effects, mean standard deviation of the distributions of allocations across the different treatments, potentially justification text analysis data

In [ ]:
# ============================================================================
# ANALYSIS CELL 1: SETUP & DATA LOAD (LLM BLOCK)
# Loads finaldataLLM.csv from your public GitHub raw URL.
# ============================================================================
!pip -q install scipy statsmodels pandas numpy

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy import stats

# EDIT THIS to your raw GitHub URL:
URL = "data url here"
df = pd.read_csv(URL)

print(f"Loaded {len(df)} rows, {df['model'].nunique()} models.")
print(f"Arms: {sorted(df['arm'].unique())}")
print(f"Draws per model x arm:\n{df.groupby(['model','arm']).size().unstack()}")

Loaded 1400 rows, 7 models.
Arms: ['control', 'history', 'placebo', 'science']
Draws per model x arm:
arm                  control  history  placebo  science
model                                                  
Claude-Haiku-4.5          50       50       50       50
DeepSeek-V4-Flash         50       50       50       50
GPT-5.6-Luna              50       50       50       50
Gemini-2.5-Flash          50       50       50       50
Grok-4.20                 50       50       50       50
Llama-4-Maverick          50       50       50       50
Qwen3-235B-Instruct       50       50       50       50


In [ ]:
# ============================================================================
# ANALYSIS CELL 2: PREP
# Convert token counts to shares (the FMLogit dependent variable) and confirm
# the constant-sum constraint holds. Set the reference categories for the
# design matrix: arm=control, paraphrase=V1, model=first alphabetically.
# ============================================================================

# Sanity: every row should sum to 100 (allow tiny float tolerance).
bad = (df[['directed','undirected','unknowable']].sum(axis=1).round(1) != 100.0).sum()
print(f"Rows not summing to 100: {bad}")

# Shares in [0,1], order: Directed, Undirected, Unknowable.
S = df[['directed','undirected','unknowable']].values / 100.0

# Categorical levels (reference = first, dropped from dummies).
ARMS   = ['science','placebo','history']          # control is reference
PARAS  = sorted(df['paraphrase_id'].unique())[1:]  # V1 reference
ORDERS = sorted(df['category_order'].unique())[1:] # first order reference
MODELS = sorted(df['model'].unique())[1:]          # first model reference

print(f"Arm contrasts vs control : {ARMS}")
print(f"Paraphrase dummies       : {PARAS}")
print(f"Order dummies            : {len(ORDERS)} levels")
print(f"Model FE dummies         : {MODELS}")

Rows not summing to 100: 0
Arm contrasts vs control : ['science', 'placebo', 'history']
Paraphrase dummies       : ['V2', 'V3', 'V4', 'V5']
Order dummies            : 5 levels
Model FE dummies         : ['DeepSeek-V4-Flash', 'GPT-5.6-Luna', 'Gemini-2.5-Flash', 'Grok-4.20', 'Llama-4-Maverick', 'Qwen3-235B-Instruct']


In [ ]:
# ============================================================================
# ANALYSIS CELL 3: DESCRIPTIVES
# Eyeball diagnostics + full summary statistics (means and SDs, all six
# outcomes) + ready-to-paste LaTeX rows for the descriptive-statistics table.
# ============================================================================
order = ['control', 'science', 'placebo', 'history']
outs  = ['directed', 'undirected', 'unknowable', 'manipcheck', 'personalnorm', 'empiricalexpectation']

print("ARM MEANS (all models pooled)")
am = df.groupby('arm')[outs].mean().round(2)
print(am.reindex(order).to_string())

print("\nARM STANDARD DEVIATIONS (all models pooled)")
asd = df.groupby('arm')[outs].std().round(2)
print(asd.reindex(order).to_string())

print("\nN per arm:", df.groupby('arm').size().reindex(order).to_dict())

print("\nDIRECTED SHARE by model x arm, with elasticity gaps")
piv = df.pivot_table('directed', 'model', 'arm', aggfunc='mean').round(2)[order]
piv['H-S gap'] = (piv['history'] - piv['science']).round(2)
piv['H-C gap'] = (piv['history'] - piv['control']).round(2)
print(piv.sort_values('H-S gap', ascending=False).to_string())

print("\nMANIPULATION CHECK by model x arm")
print(df.pivot_table('manipcheck', 'model', 'arm', aggfunc='mean').round(2)[order].to_string())

print("\nWITHIN-ARM STD of Directed share (dispersion check)")
print(df.groupby('arm')['directed'].std().round(2).reindex(order).to_string())

# ---------------------------------------------------------------------------
# LaTeX body for the descriptive-statistics table (mean with SD in parentheses)
# Paste the four printed rows over the data rows in summary_table.tex.
# ---------------------------------------------------------------------------
labs = {'control':'Control', 'science':'Science', 'placebo':'Placebo', 'history':'History'}
print("\n% ---- paste as the four data rows of Table (summary stats) ----")
for a in order:
    d = df[df.arm == a]
    cells = " & ".join(f"${d[o].mean():.2f}\\,({d[o].std():.2f})$" for o in outs)
    print(f"{labs[a]:8s} & {cells} \\\\")
print(f"% n per arm = {int(df.groupby('arm').size().reindex(order).iloc[0])} "
      f"(update the table note if this differs)")

ARM MEANS (all models pooled)
         directed  undirected  unknowable  manipcheck  personalnorm  empiricalexpectation
arm                                                                                      
control     18.09       49.17       32.73        2.12          4.12                 26.13
science     16.55       51.99       31.46        1.74          4.19                 23.47
placebo     17.30       50.89       31.80        1.40          4.28                 23.60
history     26.66       40.15       33.19        3.04          3.70                 35.42

ARM STANDARD DEVIATIONS (all models pooled)
         directed  undirected  unknowable  manipcheck  personalnorm  empiricalexpectation
arm                                                                                      
control      5.53       10.71        9.60        0.69          0.34                  5.90
science      5.78       12.47        9.99        0.47          0.42                  7.25
placebo      6.19       1

In [ ]:
# ============================================================================
# ANALYSIS CELL 4: FRACTIONAL MULTINOMIAL LOGIT (primary estimator)
# Papke-Wooldridge multinomial quasi-MLE for share outcomes that sum to 1.
# Maximizes sum_i sum_j s_ij * log(p_ij), with p a softmax over categories.
# Reference category = Unknowable. Returns coefficients + a predict function.
# ============================================================================

def build_design(data, arms=ARMS, paras=PARAS, orders=ORDERS, models=MODELS,
                 include=('arm','para','order','model')):
    """Assemble the design matrix with intercept + requested dummy blocks."""
    cols = [np.ones(len(data))]
    names = ['const']
    if 'arm' in include:
        for a in arms:   cols.append((data['arm']==a).astype(float).values);            names.append(f'arm_{a}')
    if 'para' in include:
        for p in paras:  cols.append((data['paraphrase_id']==p).astype(float).values);  names.append(f'para_{p}')
    if 'order' in include:
        for o in orders: cols.append((data['category_order']==o).astype(float).values); names.append(f'ord_{o}')
    if 'model' in include:
        for m in models: cols.append((data['model']==m).astype(float).values);          names.append(f'mod_{m}')
    return np.column_stack(cols), names

def fit_fmlogit(Xmat, Smat):
    """Fit FMLogit by QMLE. 3 categories, reference = category index 2
    (Unknowable). Uses L-BFGS-B with an analytic gradient for reliable
    convergence on the full-sample surface. Returns beta (2 x k) and result."""
    n, k = Xmat.shape
    def negqll_grad(theta):
        b = theta.reshape(2, k)
        eta = np.column_stack([Xmat @ b.T, np.zeros(n)])     # add reference col
        mx = eta.max(1, keepdims=True)
        ex = np.exp(eta - mx); p = ex / ex.sum(1, keepdims=True)
        f = -(Smat * np.log(np.clip(p, 1e-12, 1))).sum()     # negative log-QL
        g = np.empty((2, k))
        for j in range(2):                                    # analytic gradient
            g[j] = Xmat.T @ (p[:, j] - Smat[:, j])
        return f, g.ravel()
    res = minimize(negqll_grad, np.zeros(2*k), jac=True, method='L-BFGS-B',
                   options={'maxiter': 20000, 'maxfun': 50000,
                            'ftol': 1e-12, 'gtol': 1e-8})
    return res.x.reshape(2, k), res

def predict_dir(Xmat, beta):
    """Predicted Directed share (category 0) for each row."""
    eta = np.column_stack([Xmat @ beta.T, np.zeros(len(Xmat))])
    mx = eta.max(1, keepdims=True)
    p = np.exp(eta - mx); p /= p.sum(1, keepdims=True)
    return p[:, 0]

# Fit the primary model: arms + paraphrase FE + order FE + model FE.
X, names = build_design(df)
beta, res = fit_fmlogit(X, S)
print(f"FMLogit converged: {res.success} | grad norm: {np.linalg.norm(res.jac):.2e} "
      f"| log-QL: {-res.fun:.1f} | params: {beta.size}")

FMLogit converged: True | grad norm: 3.21e-04 | log-QL: -1417.0 | params: 38


In [ ]:
# ============================================================================
# ANALYSIS CELL 5: AVERAGE MARGINAL EFFECTS (the confirmatory contrasts)
# AME of each arm vs control on the Directed share, in percentage points.
# SEs by nonparametric bootstrap. Two variants:
#   - obs-level bootstrap (treats the 7 models as the population of interest;
#     model FE absorb between-model differences) -> DEFAULT
#   - model-cluster bootstrap (resamples whole models; honest if you regard the
#     7 models as a sample, but only 7 clusters so it is noisy)
# Report both; the pre-registration framing determines which you headline.
# ============================================================================

def arm_ames(data):
    """Point estimates: AME of each arm vs control on Directed share (pp)."""
    Xd, _ = build_design(data)
    b, _ = fit_fmlogit(Xd, data[['directed','undirected','unknowable']].values/100.0)
    base = Xd.copy()
    for i in range(len(ARMS)):                       # zero all arm dummies -> control
        base[:, 1+i] = 0
    p_ctrl = predict_dir(base, b)
    out = {}
    for i, a in enumerate(ARMS):
        cf = base.copy(); cf[:, 1+i] = 1
        out[a] = (predict_dir(cf, b) - p_ctrl).mean() * 100
    return out

point = arm_ames(df)

def bootstrap_ames(data, B=300, cluster=False, seed=1):
    rng = np.random.default_rng(seed)
    keep = {a: [] for a in ARMS}
    mods = data['model'].unique()
    for _ in range(B):
        if cluster:
            pick = rng.choice(mods, len(mods), replace=True)
            bs = pd.concat([data[data['model']==m] for m in pick], ignore_index=True)
        else:
            bs = data.sample(len(data), replace=True)
        try:
            a = arm_ames(bs)
            for k in ARMS: keep[k].append(a[k])
        except Exception:
            continue
    return {k: (np.std(v), np.percentile(v,[2.5,97.5])) for k,v in keep.items()}

print("AME on Directed share (percentage points), vs Control:\n")
print(f"{'arm':8s} {'AME':>8s} {'SE(obs)':>9s} {'z':>7s} {'p':>8s}")
se_obs = bootstrap_ames(df, B=300, cluster=False)
for a in ARMS:
    est = point[a]; se = se_obs[a][0]; z = est/se; p = 2*(1-stats.norm.cdf(abs(z)))
    print(f"{a:8s} {est:+8.2f} {se:9.2f} {z:7.2f} {p:8.4f}")

    # ============================================================================
# PANEL B: prompt-sensitivity & robustness (joint quasi-LR chi2 tests)
# Reuses build_design() and fit_fmlogit() from the FMLogit cell.
# ============================================================================
from scipy import stats

def _qll(Xmat, Smat, beta):
    eta = np.column_stack([Xmat @ beta.T, np.zeros(len(Xmat))])
    mx = eta.max(1, keepdims=True); p = np.exp(eta - mx); p /= p.sum(1, keepdims=True)
    return (Smat * np.log(np.clip(p, 1e-12, 1))).sum()

def _design_inter(data, interact):
    """Full design + arm x (paraphrase|order) interaction columns."""
    X, names = build_design(data)
    A = {a: (data['arm']==a).astype(float).values for a in ARMS}
    if interact == 'para':
        F = {p: (data['paraphrase_id']==p).astype(float).values for p in PARAS}
    else:
        F = {o: (data['category_order']==o).astype(float).values for o in ORDERS}
    extra = [A[a]*F[f] for a in ARMS for f in F]
    return np.column_stack([X] + extra)

def qlr(drop=None, interact=None):
    if interact:
        Xf = _design_inter(df, interact); Xr, _ = build_design(df)
    else:
        Xf, _ = build_design(df)
        inc = tuple(b for b in ('arm','para','order','model') if b != drop)
        Xr, _ = build_design(df, include=inc)
    bf, _ = fit_fmlogit(Xf, S); br, _ = fit_fmlogit(Xr, S)
    dfree = 2 * (Xf.shape[1] - Xr.shape[1])
    stat = 2 * (_qll(Xf, S, bf) - _qll(Xr, S, br))
    return stat, dfree, 1 - stats.chi2.cdf(stat, dfree)

print("PANEL B: Prompt-sensitivity and robustness (joint chi2 tests)")
print("="*62)
for lab, kw in [("Paraphrase invariance (main)", {'drop':'para'}),
                ("Category-order invariance (main)", {'drop':'order'}),
                ("Arm x Paraphrase interaction", {'interact':'para'}),
                ("Arm x Category-order interaction", {'interact':'order'})]:
    stat, dfree, p = qlr(**kw)
    verdict = "invariant" if p > 0.05 else "SENSITIVE"
    print(f"  {lab:34s} chi2({dfree:2d})={stat:7.2f}, p={p:.4f}  [{verdict}]")

AME on Directed share (percentage points), vs Control:

arm           AME   SE(obs)       z        p
science     -1.60      0.40   -4.02   0.0001
placebo     -0.78      0.40   -1.98   0.0473
history     +8.44      0.50   16.75   0.0000
PANEL B: Prompt-sensitivity and robustness (joint chi2 tests)
  Paraphrase invariance (main)       chi2( 8)=   3.23, p=0.9188  [invariant]
  Category-order invariance (main)   chi2(10)=   2.12, p=0.9954  [invariant]
  Arm x Paraphrase interaction       chi2(24)=   0.48, p=1.0000  [invariant]
  Arm x Category-order interaction   chi2(30)=   1.23, p=1.0000  [invariant]


In [ ]:
# Full three-share AMEs with SEs + stars; emits ready-to-paste LaTeX Panel A.
import numpy as np
from scipy import stats
np.random.seed(7)   # fixed seed -> reproducible SEs for the paper

def _probs(Xm, b):
    eta = np.column_stack([Xm @ b.T, np.zeros(len(Xm))])
    mx = eta.max(1, keepdims=True); p = np.exp(eta-mx); p /= p.sum(1, keepdims=True)
    return p

def ame_all(data):
    Xd, nm = build_design(data)
    aidx = [nm.index(f'arm_{a}') for a in ARMS]
    b, _ = fit_fmlogit(Xd, data[['directed','undirected','unknowable']].values/100.0)
    base = Xd.copy()
    for i in aidx: base[:, i] = 0
    p0 = _probs(base, b)
    return {a: (_probs(base.copy().__class__(np.where(np.arange(base.shape[1])==i, 1, base)), b) if False else
               (_probs(np.concatenate([base[:, :i], np.ones((len(base),1)), base[:, i+1:]], axis=1), b) - p0).mean(0)*100)
            for a, i in zip(ARMS, aidx)}

# simpler, clearer version:
def ame_all(data):
    Xd, nm = build_design(data)
    aidx = [nm.index(f'arm_{a}') for a in ARMS]
    b, _ = fit_fmlogit(Xd, data[['directed','undirected','unknowable']].values/100.0)
    base = Xd.copy()
    for i in aidx: base[:, i] = 0
    p0 = _probs(base, b)
    out = {}
    for a, i in zip(ARMS, aidx):
        cf = base.copy(); cf[:, i] = 1.0
        out[a] = (_probs(cf, b) - p0).mean(0) * 100
    return out

point = ame_all(df)
B = 400
boot = {a: [] for a in ARMS}
for _ in range(B):
    bs = df.sample(len(df), replace=True)
    try:
        r = ame_all(bs)
        for a in ARMS: boot[a].append(r[a])
    except Exception:
        pass
se = {a: np.std(np.array(boot[a]), axis=0) for a in ARMS}

def stars(p): return "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
label = {'science':'Science (Big Bang)', 'placebo':'Placebo (biomedical)', 'history':'History (testimony)'}

print("Console check [Directed, Undirected, Unknowable]:")
for a in ARMS:
    cells = [f"{point[a][j]:+.2f}({se[a][j]:.2f}){stars(2*(1-stats.norm.cdf(abs(point[a][j]/se[a][j]))))}" for j in range(3)]
    print(f"  {a:8s}: " + " | ".join(cells))

print("\n% ---- paste into Panel A ----")
for a in ARMS:
    est, s = point[a], se[a]
    def c(j):
        z = est[j]/s[j]; p = 2*(1-stats.norm.cdf(abs(z))); st = stars(p)
        return f"${est[j]:+.2f}^{{{st}}}$" if st else f"${est[j]:+.2f}$"
    print(f"{label[a]}")
    print(f" & {c(0)} & {c(1)} & {c(2)} \\\\")
    print(f" & $({s[0]:.2f})$ & $({s[1]:.2f})$ & $({s[2]:.2f})$ \\\\[2pt]")

In [ ]:
# ============================================================================
# ANALYSIS CELL 6: PROMPT-SENSITIVITY & ORDER INVARIANCE (pre-registered)
# Compares the primary model to models WITHOUT paraphrase / order blocks via a
# quasi-likelihood-ratio test. A non-significant result means dropping the block
# does not change fit -> the treatment effect is invariant to that factor.
# The headline sentence: "treatment effects are stable across the five
# paraphrases (p = .XX) and across category orderings (p = .XX)."
# ============================================================================

def qll(Xmat, Smat, beta):
    eta = np.column_stack([Xmat @ beta.T, np.zeros(len(Xmat))])
    mx = eta.max(1, keepdims=True); p = np.exp(eta-mx); p /= p.sum(1,keepdims=True)
    return (Smat * np.log(np.clip(p,1e-12,1))).sum()

def qlr_test(drop_block):
    """Quasi-LR: full model vs model with `drop_block` removed."""
    Xf, nf = build_design(df)
    bf, _ = fit_fmlogit(Xf, S); llf = qll(Xf, S, bf)
    inc = tuple(b for b in ('arm','para','order','model') if b != drop_block)
    Xr, nr = build_design(df, include=inc)
    br, _ = fit_fmlogit(Xr, S); llr = qll(Xr, S, br)
    dfree = 2 * (Xf.shape[1] - Xr.shape[1])          # 2 non-ref categories
    stat = 2 * (llf - llr)
    p = 1 - stats.chi2.cdf(stat, dfree)
    return stat, dfree, p

for block, label in [('para','Paraphrase invariance'), ('order','Order invariance')]:
    stat, dfree, p = qlr_test(block)
    verdict = "INVARIANT (good)" if p > 0.05 else "sensitive (investigate)"
    print(f"{label:22s}: chi2({dfree})={stat:.2f}, p={p:.4f}  -> {verdict}")

In [ ]:
# ============================================================================
# ANALYSIS CELL 7: CLOGG Z — CROSS-BLOCK ELASTICITY GAP
#   Z = (AME_human - AME_LLM) / sqrt(SE_human^2 + SE_LLM^2)
# Human estimates from the human-notebook FMLogit (400 bootstrap reps,
# observation-level), pasted in as constants.
# ============================================================================
import numpy as np
from scipy import stats

def clogg_z(ame1, se1, ame2, se2):
    z = (ame1 - ame2) / np.sqrt(se1**2 + se2**2)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p

# Human block: Directed-share AMEs (pp) and bootstrap SEs, corrected data
HUMAN = {'science': (-2.39, 3.82),
         'placebo': (+0.95, 4.10),
         'history': (+1.39, 2.60)}

print(f"{'arm':<10} {'Human AME (SE)':>18} {'LLM AME (SE)':>18} {'Z':>8} {'p':>8}")
for a in ['science', 'placebo', 'history']:
    ame_L = point[a][0]        # index 0 = Directed
    se_L  = se_obs[a][0]
    ame_H, se_H = HUMAN[a]
    z, p = clogg_z(ame_H, se_H, ame_L, se_L)
    tag = "  <-- PRIMARY (prereg)" if a == 'science' else ""
    print(f"{a:<10} {ame_H:+8.2f} ({se_H:.2f})   {ame_L:+8.2f} ({se_L:.2f}) "
          f"{z:+8.2f} {p:8.4f}{tag}")

In [ ]:
# ============================================================================
# FINAL CANONICAL RUN — LLM bootstrap + cross-block tests, single source
# Reruns the observation-level bootstrap once (fixed seed) and derives EVERY
# reported number from this output: LLM AMEs + SEs (all three shares),
# Clogg Z vs human block, nominal p, and Holm-adjusted p for the secondary
# family. Populate all tables from this printout only.
# ============================================================================
import numpy as np
from scipy import stats

SEED = 307302     # <-- must match whatever seed the manuscript claims
B    = 400

# ---- Human block constants (from human notebook, corrected data, final run)
HUMAN = {'science': (-2.39, 3.82),
         'placebo': (+0.95, 4.10),
         'history': (+1.39, 2.60)}

# ---- helper: predicted shares for all 3 categories
def predict_all(Xmat, beta):
    eta = np.column_stack([Xmat @ beta.T, np.zeros(len(Xmat))])
    mx = eta.max(1, keepdims=True)
    p = np.exp(eta - mx); p /= p.sum(1, keepdims=True)
    return p

def ame_all(data, beta):
    """AME (pp) on all three shares, each arm vs control, full design."""
    base = data.copy(); base['arm'] = 'control'
    Xb, _ = build_design(base)
    p0 = predict_all(Xb, beta).mean(0)
    out = {}
    for a in ARMS:
        cf = data.copy(); cf['arm'] = a
        Xa, _ = build_design(cf)
        out[a] = (predict_all(Xa, beta).mean(0) - p0) * 100
    return out

# ---- point estimates
S = df[['directed','undirected','unknowable']].values / 100.0
X, _ = build_design(df)
beta_hat, res = fit_fmlogit(X, S)
assert res.success, "point fit failed"
point = ame_all(df, beta_hat)

# ---- bootstrap (observation-level, fixed seed)
rng = np.random.default_rng(SEED)
boot = {a: [] for a in ARMS}
n = len(df); fails = 0
for b in range(B):
    idx = rng.integers(0, n, n)
    d_b = df.iloc[idx].reset_index(drop=True)
    S_b = d_b[['directed','undirected','unknowable']].values / 100.0
    X_b, _ = build_design(d_b)
    beta_b, r_b = fit_fmlogit(X_b, S_b)
    if not r_b.success:
        fails += 1; continue
    a_b = ame_all(d_b, beta_b)
    for a in ARMS:
        boot[a].append(a_b[a])
print(f"bootstrap: {B - fails}/{B} successful | seed={SEED}")

SE = {a: np.array(boot[a]).std(axis=0, ddof=1) for a in ARMS}

# ---- output block 1: LLM Panel A (all three shares)
CATS = ['Directed', 'Undirected', 'Unknowable']
print("\n=== LLM Panel A (canonical) ===")
for a in ARMS:
    z = point[a] / SE[a]
    p = 2 * (1 - stats.norm.cdf(np.abs(z)))
    stars = ["***" if v < .001 else "**" if v < .01 else "*" if v < .05 else ""
             for v in p]
    print(f"{a:<8} " + " | ".join(
        f"{CATS[j]}: {point[a][j]:+.2f}{stars[j]} ({SE[a][j]:.2f})"
        for j in range(3)))

# ---- output block 2: Clogg Z + Holm (Directed share)
def clogg_z(aH, sH, aL, sL):
    z = (aH - aL) / np.sqrt(sH**2 + sL**2)
    return z, 2 * (1 - stats.norm.cdf(abs(z)))

print("\n=== Cross-block Clogg Z (Directed share) ===")
pvals = {}
for a in ['science', 'placebo', 'history']:
    aH, sH = HUMAN[a]
    aL, sL = point[a][0], SE[a][0]
    z, p = clogg_z(aH, sH, aL, sL)
    pvals[a] = p
    tag = "  <-- PRIMARY" if a == 'science' else ""
    print(f"{a:<8} Human {aH:+.2f} ({sH:.2f}) | LLM {aL:+.2f} ({sL:.2f}) | "
          f"Z={z:+.2f}, p={p:.4f}{tag}")

print("\n=== Holm adjustment, secondary family (placebo, history) ===")
sec = {a: pvals[a] for a in ['placebo', 'history']}
order = sorted(sec, key=sec.get)
adj_prev = 0.0
for i, a in enumerate(order):
    adj = min(max((len(order) - i) * sec[a], adj_prev), 1.0)
    adj_prev = adj
    print(f"  {a}: nominal p={sec[a]:.4f}, Holm-adjusted p={adj:.4f}")

Figures

In [ ]:
# ============================================================================
# VISUALIZATION CELL: allocation bar charts and histograms
# Assumes df is already loaded (model, arm, directed, undirected, unknowable).
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

arms   = ["control", "science", "placebo", "history"]
cats   = ["directed", "undirected", "unknowable"]
labels = ["Directed", "Undirected", "Unknowable"]
colors = ["#534AB7", "#1D9E75", "#888780"]

# ---------- Chart 1: grouped mean +/- SD by arm ----------
means = df.groupby("arm")[cats].mean().reindex(arms)
sds   = df.groupby("arm")[cats].std().reindex(arms)
x = np.arange(len(arms)); w = 0.26
fig, ax = plt.subplots(figsize=(9, 5.2))
for i, (c, lab, col) in enumerate(zip(cats, labels, colors)):
    ax.bar(x + (i-1)*w, means[c], w, yerr=sds[c], capsize=4,
           label=lab, color=col, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels([a.capitalize() for a in arms])
ax.set_ylabel("Mean tokens allocated"); ax.set_ylim(0, 80)
ax.set_title("Mean token allocation by treatment arm (±1 SD), pooled across models")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.08))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# ---------- Chart 2: pooled histogram of all three categories ----------
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(0, 101, 5)
for c, lab, col in zip(cats, labels, colors):
    ax.hist(df[c], bins=bins, alpha=0.55, label=lab, color=col, edgecolor="white")
ax.set_xlabel("Tokens allocated"); ax.set_ylabel("Frequency")
ax.set_title("Distribution of token allocations, pooled across all models and arms")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# ---------- Chart 3: Directed-share histogram faceted by arm ----------
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey=True)
for ax, arm in zip(axes.ravel(), arms):
    d = df[df["arm"] == arm]["directed"]
    ax.hist(d, bins=np.arange(0, 61, 4), color="#534AB7", edgecolor="white")
    ax.axvline(d.mean(), color="#D85A30", lw=2, ls="--")
    ax.set_title(f"{arm.capitalize()}  (mean={d.mean():.1f})")
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes[1, :]: ax.set_xlabel("Directed tokens")
for ax in axes[:, 0]: ax.set_ylabel("Frequency")
fig.suptitle("Directed-share distribution by treatment arm (dashed = mean)", y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# VISUALIZATION CELL (revised): clustered mean bar + percentage histogram
# Assumes df is loaded (model, arm, directed, undirected, unknowable).
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

arms    = ["control", "science", "placebo", "history"]
arm_col = {"control": "#888780", "science": "#1D9E75",
           "placebo": "#D85A30", "history": "#534AB7"}   # one color per arm
cats    = ["directed", "undirected", "unknowable"]
labels  = ["Directed", "Undirected", "Unknowable"]
cat_col = ["#534AB7", "#1D9E75", "#888780"]

# ---------- Chart A: clustered mean allocation by treatment (sums to 100), ±1 SD
means = df.groupby("arm")[cats].mean().reindex(arms)
sds   = df.groupby("arm")[cats].std().reindex(arms)     # standard deviation
x = np.arange(len(arms)); w = 0.26
fig, ax = plt.subplots(figsize=(9, 5.4))
for i, (c, lab, col) in enumerate(zip(cats, labels, cat_col)):
    ax.bar(x + (i-1)*w, means[c], w, yerr=sds[c], capsize=4,
           label=lab, color=col, edgecolor="white")
for j, a in enumerate(arms):                             # confirm sum-to-100
    ax.text(j, 72, f"Σ={means.loc[a].sum():.0f}", ha="center", fontsize=9, color="#555")
ax.set_xticks(x); ax.set_xticklabels([a.capitalize() for a in arms])
ax.set_ylabel("Mean tokens allocated"); ax.set_ylim(0, 78)
ax.set_title("Mean token allocation by treatment arm (±1 SD); three means sum to 100")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.09))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# ---------- Chart B: Directed distribution by arm, one color per arm, % of arm sample
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5), sharex=True, sharey=True)
bins = np.arange(0, 61, 4)
for ax, arm in zip(axes.ravel(), arms):
    d = df[df["arm"] == arm]["directed"]
    wts = np.ones(len(d)) / len(d) * 100                 # percent of THIS arm's sample
    ax.hist(d, bins=bins, weights=wts, color=arm_col[arm], edgecolor="white")
    ax.axvline(d.mean(), color="#111", lw=2, ls="--")
    ax.set_title(f"{arm.capitalize()}  (mean={d.mean():.1f})", color=arm_col[arm])
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes[1, :]: ax.set_xlabel("Directed tokens")
for ax in axes[:, 0]: ax.set_ylabel("% of arm sample")
fig.suptitle("Directed-share distribution by treatment arm (dashed = mean)", y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# FIGURE: clustered mean allocation by arm (±1 SD) with omnibus test annotation
# Welch's ANOVA (unequal-variance robust) across arms for each share, plus
# Games-Howell post-hoc to localize the difference to History.
# ============================================================================
!pip -q install pingouin
import numpy as np, pandas as pd, pingouin as pg
import matplotlib.pyplot as plt

arms   = ["control", "science", "placebo", "history"]
cats   = ["directed", "undirected", "unknowable"]
labels = ["Directed", "Undirected", "Unknowable"]
cat_col = ["#534AB7", "#1D9E75", "#888780"]

means = df.groupby("arm")[cats].mean().reindex(arms)
sds   = df.groupby("arm")[cats].std().reindex(arms)

# --- Welch's ANOVA per share (unequal variances -> Welch, not classic F) ---
res = {}
for dv in cats:
    w = pg.welch_anova(data=df, dv=dv, between="arm")
    res[dv] = (int(w.ddof1[0]), w.ddof2[0], w.F[0], w.p_unc[0])

# --- Games-Howell post-hoc on Directed (unequal-variance pairwise) ---
gh = pg.pairwise_gameshowell(data=df, dv="directed", between="arm")
print(gh[["A","B","diff","pval"]].round(4).to_string(index=False))

def pstr(p): return "p<0.001" if p < 0.001 else f"p={p:.3f}"

x = np.arange(len(arms)); wbar = 0.26
fig, ax = plt.subplots(figsize=(9.4, 5.9))
for i, (c, lab, col) in enumerate(zip(cats, labels, cat_col)):
    ax.bar(x + (i-1)*wbar, means[c], wbar, yerr=sds[c], capsize=4,
           label=lab, color=col, edgecolor="white")
for j, a in enumerate(arms):
    ax.text(j, 84, f"$\\Sigma$={means.loc[a].sum():.0f}", ha="center", fontsize=8.5, color="#888")

xh = 3 - wbar
ax.annotate("History differs from all\nother arms (Games-Howell,\n$p<0.001$)",
            xy=(xh, means.loc["history","directed"] + sds.loc["history","directed"] + 1.5),
            xytext=(xh-0.15, 70), ha="center", fontsize=8.5, color="#333",
            arrowprops=dict(arrowstyle="->", color="#333", lw=1.2))

box = ("Welch's ANOVA across arms\n"
       f"Directed:   $F({res['directed'][0]},{res['directed'][1]:.0f})$={res['directed'][2]:.1f}, {pstr(res['directed'][3])}\n"
       f"Undirected: $F({res['undirected'][0]},{res['undirected'][1]:.0f})$={res['undirected'][2]:.1f}, {pstr(res['undirected'][3])}\n"
       f"Unknowable: $F({res['unknowable'][0]},{res['unknowable'][1]:.0f})$={res['unknowable'][2]:.2f}, {pstr(res['unknowable'][3])} (n.s.)")
ax.text(0.015, 0.975, box, transform=ax.transAxes, va="top", ha="left",
        fontsize=8, family="monospace",
        bbox=dict(boxstyle="round,pad=0.5", fc="#F4F3EE", ec="#CCC"))

ax.set_xticks(x); ax.set_xticklabels([a.capitalize() for a in arms])
ax.set_ylabel("Mean tokens allocated"); ax.set_ylim(0, 92)
ax.set_title("Mean token allocation by treatment arm (±1 SD)")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.08))
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# VISUALIZATION CELL: clustered means with significance annotation
# Adds a Welch t-test bracket (History vs Control on Directed) plus a stats box.
# Assumes df is loaded (model, arm, directed, undirected, unknowable).
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

arms    = ["control", "science", "placebo", "history"]
cats    = ["directed", "undirected", "unknowable"]
labels  = ["Directed", "Undirected", "Unknowable"]
cat_col = ["#534AB7", "#1D9E75", "#888780"]

means = df.groupby("arm")[cats].mean().reindex(arms)
sds   = df.groupby("arm")[cats].std().reindex(arms)      # standard deviation

# ---- Welch t-test on raw Directed tokens: History vs Control ----
dir_hist = df[df.arm == "history"]["directed"]
dir_ctrl = df[df.arm == "control"]["directed"]
t_hc, p_hc = stats.ttest_ind(dir_hist, dir_ctrl, equal_var=False)
delta_hc = dir_hist.mean() - dir_ctrl.mean()
def stars(p): return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"

# ---- Plot ----
x = np.arange(len(arms)); w = 0.26
fig, ax = plt.subplots(figsize=(9.2, 5.8))
for i, (c, lab, col) in enumerate(zip(cats, labels, cat_col)):
    ax.bar(x + (i-1)*w, means[c], w, yerr=sds[c], capsize=4,
           label=lab, color=col, edgecolor="white")
for j, a in enumerate(arms):
    ax.text(j, 82, f"Σ={means.loc[a].sum():.0f}", ha="center", fontsize=9, color="#777")

# Significance bracket connecting the two Directed bars (Control -> History)
xc, xh, yb = 0 - w, 3 - w, 70
ax.plot([xc, xc, xh, xh], [yb-2, yb, yb, yb-2], lw=1.5, color="#111")
ax.annotate(f"Directed:  Δ = {delta_hc:+.1f} tokens   {stars(p_hc)}",
            xy=((xc+xh)/2, yb), xytext=((xc+xh)/2, yb+2.5),
            ha="center", fontsize=11, fontweight="bold", color="#111")

# Stats box: descriptive t-test + confirmatory FMLogit AME
box = (f"History vs Control (Welch t)\n"
       f"Δ Directed = {delta_hc:+.1f}  (t={t_hc:.2f}, p<0.001)\n"
       f"FMLogit AME = +9.5 pp  (z=4.38, p<0.001)")   # update AME from Cell 5 at full n
ax.text(0.015, 0.97, box, transform=ax.transAxes, va="top", ha="left",
        fontsize=8.5, family="monospace",
        bbox=dict(boxstyle="round,pad=0.5", fc="#F4F3EE", ec="#CCC"))

ax.set_xticks(x); ax.set_xticklabels([a.capitalize() for a in arms])
ax.set_ylabel("Mean tokens allocated"); ax.set_ylim(0, 90)
ax.set_title("Mean token allocation by treatment arm (±1 SD); three means sum to 100")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.09))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# FIGURE: cross-model elasticity dumbbell (Directed share, Control -> History)
# One row per model, sorted by swing (History minus Control). Each model shows
# its untreated baseline and its treated allocation joined by a connector whose
# length is the treatment effect. Assumes df is loaded.
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

# Directed-share means by model for the two arms that define the connector.
piv = df.pivot_table("directed", "model", "arm", aggfunc="mean")
piv["swing"] = piv["history"] - piv["control"]          # History vs Control
piv = piv.sort_values("swing")                          # smallest swing at bottom

y = np.arange(len(piv))
ctrl_col, hist_col = "#888780", "#534AB7"

fig, ax = plt.subplots(figsize=(9, 5.4))
for yi, (_, r) in zip(y, piv.iterrows()):               # connectors
    ax.plot([r["control"], r["history"]], [yi, yi], color="#C9C7BE", lw=2.5, zorder=1)
ax.scatter(piv["control"], y, s=70, color=ctrl_col, zorder=2, label="Control")
ax.scatter(piv["history"], y, s=70, color=hist_col, zorder=2, label="History")
for yi, (_, r) in zip(y, piv.iterrows()):               # swing label at History end
    ax.annotate(f"+{r['swing']:.1f}", (r["history"], yi), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=8.5, color=hist_col)

ax.axvline(piv["control"].mean(), color=ctrl_col, ls=":", lw=1, alpha=0.6)  # mean baseline
ax.set_yticks(y); ax.set_yticklabels(piv.index)
ax.set_xlabel("Directed-share allocation (tokens)")
ax.set_title("Testimony shifts the Directed share more in some models than others")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.12)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# CROSS-MODEL HETEROGENEITY: (1) arm x model interaction test; (2) per-model
# History AME on the Directed share with bootstrap 95% CIs. Outputs a table.
# Reuses ARMS/PARAS/ORDERS/MODELS and the L-BFGS-B fit from the analysis cells.
# ============================================================================
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy import stats
np.random.seed(11)

MODELS_ALL = sorted(df['model'].unique())
S = df[['directed','undirected','unknowable']].values / 100.0

def design(data, include=('arm','para','order','model'), interact=None):
    cols = [np.ones(len(data))]
    A  = {a: (data['arm']==a).astype(float).values for a in ARMS}
    Md = {m: (data['model']==m).astype(float).values for m in MODELS}
    if 'arm'   in include: cols += [A[a] for a in ARMS]
    if 'para'  in include: cols += [(data['paraphrase_id']==p).astype(float).values for p in PARAS]
    if 'order' in include: cols += [(data['category_order']==o).astype(float).values for o in ORDERS]
    if 'model' in include: cols += [Md[m] for m in MODELS]
    if interact == 'armmodel':
        for a in ARMS:
            for m in MODELS: cols.append(A[a]*Md[m])
    return np.column_stack(cols)

def fit(Xm, Sm):
    n, k = Xm.shape
    def fg(t):
        b = t.reshape(2, k); eta = np.column_stack([Xm@b.T, np.zeros(n)])
        mx = eta.max(1, keepdims=True); ex = np.exp(eta-mx); p = ex/ex.sum(1, keepdims=True)
        f = -(Sm*np.log(np.clip(p,1e-12,1))).sum()
        g = np.empty((2,k))
        for j in range(2): g[j] = Xm.T@(p[:,j]-Sm[:,j])
        return f, g.ravel()
    r = minimize(fg, np.zeros(2*k), jac=True, method='L-BFGS-B',
                 options={'maxiter':20000,'maxfun':50000,'ftol':1e-12,'gtol':1e-8})
    return r.x.reshape(2,k), r

def qll(Xm, Sm, b):
    eta = np.column_stack([Xm@b.T, np.zeros(len(Xm))]); mx = eta.max(1, keepdims=True)
    p = np.exp(eta-mx); p /= p.sum(1, keepdims=True)
    return (Sm*np.log(np.clip(p,1e-12,1))).sum()

# ---- (1) Arm x Model interaction joint test (is the effect heterogeneous?) ----
Xf = design(df, interact='armmodel'); bf, _ = fit(Xf, S)
Xr = design(df);                      br, _ = fit(Xr, S)
dfree = 2*(Xf.shape[1]-Xr.shape[1]); stat = 2*(qll(Xf,S,bf)-qll(Xr,S,br))
p_int = 1 - stats.chi2.cdf(stat, dfree)
print(f"Arm x Model interaction (heterogeneity test): "
      f"chi2({dfree}) = {stat:.2f}, p = {p_int:.3g}\n")

# ---- (2) Per-model History AME on Directed, bootstrap 95% CI ----
def hist_ame(data, model):
    d  = data[data.model == model]
    Xd = design(d, include=('arm',))                 # arm-only within one model
    b, _ = fit(Xd, d[['directed','undirected','unknowable']].values/100.0)
    base = Xd.copy(); base[:, 1:1+len(ARMS)] = 0
    def pdir(X):
        eta = np.column_stack([X@b.T, np.zeros(len(X))]); mx = eta.max(1, keepdims=True)
        pp = np.exp(eta-mx); pp /= pp.sum(1, keepdims=True); return pp[:,0]
    hi = base.copy(); hi[:, 1+ARMS.index('history')] = 1
    return (pdir(hi) - pdir(base)).mean()*100

rows = []
for m in MODELS_ALL:
    pt = hist_ame(df, m)
    dm = df[df.model == m]
    bs = []
    for _ in range(300):
        r = dm.sample(len(dm), replace=True)
        try: bs.append(hist_ame(r, m))
        except Exception: pass
    lo, hi = np.percentile(bs, [2.5, 97.5]); se = np.std(bs)
    pv = 2*(1 - stats.norm.cdf(abs(pt/se)))
    rows.append((m, pt, se, lo, hi, pv))

res = pd.DataFrame(rows, columns=['model','AME','SE','CI_lo','CI_hi','p']).sort_values('AME', ascending=False)
def st(p): return "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
res['sig'] = res['p'].map(st)
print("Per-model History AME on Directed share (percentage points), bootstrap 95% CI:")
print(res.assign(AME=res.AME.round(2), SE=res.SE.round(2),
                 CI_lo=res.CI_lo.round(2), CI_hi=res.CI_hi.round(2),
                 p=res.p.round(4)).to_string(index=False))

In [ ]:
print(df.groupby('arm')[['directed','undirected','unknowable']].agg(['mean','std']).round(2))